# Metadata tutorial with YAML

In [18]:
import yaml
import numpy as np
import importlib
import os
import sys 
cwd = os.getcwd()
itop=cwd.find("cgshells/")+len("cgshells")
PROJECT_ROOT = cwd[:itop]
sys.path.insert(0, PROJECT_ROOT )

import utils.run_manager as rm
from utils.run_manager import PROJECT_ROOT, lmpunity, lmplocal
from utils.readsim import ReadSim

version = "v1"
curvsim = importlib.import_module(f"utils.curvsim.{version}")
Curvamer2D = rm.load_class(version, "curvamer2d", "Curvamer2D")
Curvamer3D = rm.load_class(version, "curvamer3d", "Curvamer3D")
versionpath = "/".join(curvsim.__name__.split("."))
DATASCRIPTS = f"{PROJECT_ROOT}/{versionpath}/DataScripts"

# Writing

In [2]:
##### PARTICLE #####
### Geometry
dimension = 2
dcore = 1.0    # hard core diameter of beads (dcore approx thickness of one DNA helix 3.5nm)
t0 = 1.0 * dcore    # structural thickness
wx = 56 * dcore    # curvamer width (arclength along midline)
r0 = 34 * dcore
Nbeads = 90    # number of beads per layer (2Nbeads is beads per curvamer)
fraction = 1/3    # middle patch of beads has width = fraction * wx

if r0 == "flat":
    k_0 = 0
else:
    k_0 = 1/r0

### Elasticity
kh = 10000
nu = 0.3
d = wx/(Nbeads-1)   # bead spacing
kvkh = 1 #(12 * t0**2 * (1-nu)) / (3 * d**2 * (1-nu) - 4 * t0**2)
kckh = (3 * (d**2 + t0**2) * (1-nu))/(2 * t0**2)

### Interactions
pair_ints = "repulsive"
sigma = 1*dcore
epsilon = 0.012938329803772514
shift = dcore - 2**(1/6)*sigma     # shift factor to make sure lj minimum is at dcore
ljcut = 5*sigma #t0 + 2*dcore               # cutoff distance for attractive lj potential
wcacut = dcore    # cutoff distance for repulsive wca potential
# softsigma = 1*sigma
# softepsilon = 1 * epsilon
softsigma = 5*sigma
softepsilon = 5e-8 * epsilon
softshift = 0 #softcore - 2**(1/6)*softsigma
softcut = 2**(1/6) * softsigma

##### SIMULATION #####
config = "dispersed"
simtype = "md"

### Concentration
nshells = 256
phi = 0.01    # concentration of molecules (area fraction) - only for MD
Nx = 4    # number of particle columns for initial config (applies if simpath_old = "none")
Ny = int(nshells/Nx)

### Dynamics/Minimization Settings
simtype = "md"
datagz = True
trajgz = False
screen = True    # output lammps log to screen
minstyle = "cg"
etol = 1e-14
maxiter = 100000
Tstart = 1.0
Tstop = Tstart
Tdamp = 10
seed = 15298
timestep = 0.0005
runsteps = 10000
dumpfreq = 100 # maxiter
thermofreq = 100
# force  = 300
px = 2
py = 2
pz = 1

### Simulation Box
v0 = wx * (t0 + dcore)
lbox = np.sqrt(nshells * v0 / phi)    # side length of (square) sim box to give proper concentration
xlo = -lbox/2
xhi = lbox/2
ylo = -lbox/2
yhi = lbox/2
zlo = -0.5
zhi = 0.5

### Simulation Directories
load_simpath = "none" # location of trajectory file to load in ("none" will create initial melt lattice)
simpath = "test" #"InitialConfigs/Nbeads-{}-Nmols-{}/phi-{:.3f}".format(N,Nmols,phi)

### Computation
# computer = "local"
computer = "unity"
nnodes = 1
mem = 16 #GB
tlim_hrs = 0
tlim_min = 10
partition = "cpu-preempt"
jobname = os.path.splitext(os.path.basename(sys.argv[0]))[0]
requested_walltime = f'{tlim_hrs:02d}:{tlim_min:02d}:00'


params = {
 
    'particle':{
        'geometry':{
            'dimension':dimension,
            'dcore':dcore,
            't0':t0,
            'wx':wx,
            'r0':r0,
            'Nbeads':Nbeads,
            'fraction':fraction
        },
        'elasticity':{
            'nu':nu,
            'kh':kh,
            'kckh':kckh,
            'kvkh':kvkh
        },
        'interactions':{
            'sigma':sigma,
            'epsilon':epsilon,
            'shift':shift,
            'ljcut':ljcut,
            'wcacut':wcacut,
            'softsigma':softsigma,
            'softepsilon':softepsilon,
            'softshift':softshift,
            'softcut':softcut,
            'pair_ints':pair_ints
        },
    },
    
    'simulation':{
        'simtype':simtype,
        'config':config,
        'nshells':nshells,
        'phi':phi,
        'xlo':float(xlo),
        'xhi':float(xhi),
        'ylo':float(ylo),
        'yhi':float(yhi),
        'zlo':float(zlo),
        'zhi':float(zhi),
        'simbox_x':float(xhi-xlo),
        'simbox_y':float(yhi-ylo),
        'simbox_z':float(zhi-zlo),
        'thermofreq':thermofreq,
        'dumpfreq':dumpfreq,
        'Tstart':Tstart,
        'Tstop':Tstop,
        'Tdamp':Tdamp,
        'seed':seed,
        'timestep':timestep,
        'runsteps':runsteps,
        'minstyle':minstyle,
        'etol':etol,
        'maxiter':maxiter
        
    },
    
    'logistics':{
        'computer':computer,
        'jobname':jobname,
        'simpath':simpath,
        'load_simpath':load_simpath,
        'nnodes':nnodes,
        'cpus':px*py*pz,
        'mem':mem,
        'partition':partition,
        'requested_walltime':requested_walltime,
        'run_counter':0
        

        
    }
}

# Write YAML metadata
with open(PROJECT_ROOT / "reference/metadata.yaml", "w") as f:
    yaml.dump(params, f, sort_keys=False)

# Reading

In [3]:
config_path = PROJECT_ROOT / "reference/metadata.yaml"
with open(config_path) as f:
    meta = yaml.safe_load(f)

In [4]:
meta

{'particle': {'geometry': {'dimension': 2,
   'dcore': 1.0,
   't0': 1.0,
   'wx': 56.0,
   'r0': 34.0,
   'Nbeads': 90,
   'fraction': 0.3333333333333333},
  'elasticity': {'nu': 0.3,
   'kh': 10000,
   'kckh': 1.4657050877414468,
   'kvkh': 1},
  'interactions': {'sigma': 1.0,
   'epsilon': 0.012938329803772514,
   'shift': -0.12246204830937302,
   'ljcut': 5.0,
   'wcacut': 1.0,
   'softsigma': 5.0,
   'softepsilon': 6.469164901886257e-10,
   'softshift': 0,
   'softcut': 5.612310241546865,
   'pair_ints': 'repulsive'}},
 'simulation': {'simtype': 'md',
  'config': 'dispersed',
  'nshells': 256,
  'phi': 0.01,
  'xlo': -846.640419540669,
  'xhi': 846.640419540669,
  'ylo': -846.640419540669,
  'yhi': 846.640419540669,
  'zlo': -0.5,
  'zhi': 0.5,
  'simbox_x': 1693.280839081338,
  'simbox_y': 1693.280839081338,
  'simbox_z': 1.0,
  'thermofreq': 100,
  'dumpfreq': 100,
  'Tstart': 1.0,
  'Tstop': 1.0,
  'Tdamp': 10,
  'seed': 15298,
  'timestep': 0.0005,
  'runsteps': 10000,
  'mins

In [32]:
meta['simulation']['xlo']

-846.640419540669

In [5]:
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00',
 'run_counter': 0}

In [37]:
int(False)

0

# Appending to existing file

In [91]:
config_path = PROJECT_ROOT / "reference/metadata.yaml"

params = {
    
    'walltime':'01:00:00',
    'run_counter':0
}

with open(config_path) as f:
    meta = yaml.safe_load(f)
    

In [95]:
meta

{'particle': {'geometry': {'dimension': 2,
   'dcore': 1.0,
   't0': 1.0,
   'wx': 56.0,
   'r0': 34.0,
   'Nbeads': 90,
   'fraction': 0.3333333333333333},
  'elasticity': {'nu': 0.3,
   'kh': 10000,
   'kckh': 1.4657050877414468,
   'kvkh': 1},
  'interactions': {'sigma': 1.0,
   'epsilon': 0.012938329803772514,
   'shift': -0.12246204830937302,
   'ljcut': 5.0,
   'wcacut': 1.0,
   'softsigma': 5.0,
   'softepsilon': 6.469164901886257e-10,
   'softshift': 0,
   'softcut': 5.612310241546865,
   'pair_ints': 'repulsive'}},
 'simulation': {'simtype': 'md',
  'config': 'dispersed',
  'nshells': 256,
  'phi': 0.01,
  'xlo': -846.640419540669,
  'xhi': 846.640419540669,
  'ylo': -846.640419540669,
  'yhi': 846.640419540669,
  'zlo': -0.5,
  'zhi': 0.5,
  'simbox_x': 1693.280839081338,
  'simbox_y': 1693.280839081338,
  'simbox_z': 1.0,
  'thermofreq': 100,
  'dumpfreq': 100,
  'Tstart': 1.0,
  'Tstop': 1.0,
  'Tdamp': 10,
  'seed': 15298,
  'timestep': 0.0005,
  'runsteps': 10000,
  'mins

In [94]:
meta['post'].update({'energies':[]})

In [93]:
meta.setdefault('post',{})

{}

In [97]:
meta['post']['energies'].append(0)

In [98]:
meta['post']['energies']

[0]

In [81]:
meta['notes']

KeyError: 'notes'

In [11]:
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00',
 'run_counter': 0,
 'walltime': '01:00:00'}

In [10]:
meta['logistics'].update({'walltime':'01:00:00'})

In [68]:
params = {'logistics':{'walltime':'01:00:00'}}

In [73]:
meta.update(params)

In [74]:
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00'}

In [53]:
meta['logistics']['run_counter']+=1

In [27]:
meta['logistics'].setdefault('jobids',[])

[]

In [54]:
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00',
 'walltime': '01:00:00',
 'run_counter': 2}

In [29]:
meta['logistics'].setdefault('jobids',[]).append('12345')

In [30]:
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00',
 'walltime': '01:00:00',
 'jobids': ['12345']}

In [64]:
jobid = "1002"

In [65]:
if jobid != None:
    print(1)

1
